In [0]:
# Load data from Bronze layer (raw ingested data)
df = spark.read.table("bronze_rides")

df.show(5)

+----------+-------------------+------------+---------------+------------+-------------+-------------------+-----------------+--------+--------+---------------------+----------------------+-------------------+--------------------+----------------+-----------------+-------------+-------------+-------------+---------------+--------------+--------------------+
|      date|               time|  booking_id| booking_status| customer_id| vehicle_type|    pickup_location|    drop_location|avg_vtat|avg_ctat|cancelled_by_customer|customer_cancel_reason|cancelled_by_driver|driver_cancel_reason|incomplete_rides|incomplete_reason|booking_value|ride_distance|driver_rating|customer_rating|payment_method|      ingestion_time|
+----------+-------------------+------------+---------------+------------+-------------+-------------------+-----------------+--------+--------+---------------------+----------------------+-------------------+--------------------+----------------+-----------------+-------------+-

In [0]:
# Remove duplicate records to ensure data consistency
df = df.dropDuplicates()

In [0]:
# Convert string "null" into actual NULL values
df = df.na.replace("null", None)

In [0]:
from pyspark.sql.functions import col

# Convert columns to numeric types for calculations
df = df.withColumn("booking_value", col("booking_value").cast("double")) \
       .withColumn("ride_distance", col("ride_distance").cast("double")) \
       .withColumn("driver_rating", col("driver_rating").cast("double")) \
       .withColumn("customer_rating", col("customer_rating").cast("double"))

In [0]:
from pyspark.sql.functions import split, concat_ws, to_timestamp

# Extract only the time part (ignore wrong date in 'time' column)
df = df.withColumn(
    "clean_time",
    split(col("time"), " ").getItem(1)
)

# Combine correct date + extracted time into one datetime column
df = df.withColumn(
    "booking_datetime",
    to_timestamp(
        concat_ws(" ", col("date"), col("clean_time")),
        "yyyy-MM-dd HH:mm:ss"
    )
)

In [0]:
# Remove incorrect and temporary columns
df = df.drop("time", "clean_time")

In [0]:
from pyspark.sql.functions import year, month, dayofweek, hour, when

# Create time-based and business features
df = df.withColumn("year", year("booking_datetime")) \
       .withColumn("month", month("booking_datetime")) \
       .withColumn("day_of_week", dayofweek("booking_datetime")) \
       .withColumn("hour", hour("booking_datetime")) \
       .withColumn(
           "is_completed",
           when(col("booking_status") == "Completed", 1).otherwise(0)
       ).withColumn(
           "is_cancelled",
           when(col("booking_status").isin("Cancelled", "No Driver Found"), 1).otherwise(0)
       )

In [0]:
# Check datetime correctness
df.select("date", "booking_datetime").show(5, False)

# Check booking status distribution
df.groupBy("booking_status").count().show()

# Revenue sanity check
df.selectExpr(
    "sum(booking_value) as total_revenue",
    "avg(booking_value) as avg_value"
).show()

+----------+-------------------+
|date      |booking_datetime   |
+----------+-------------------+
|2024-03-23|2024-03-23 12:29:38|
|2024-11-29|2024-11-29 18:01:39|
|2024-08-23|2024-08-23 08:56:10|
|2024-10-21|2024-10-21 17:17:25|
|2024-09-16|2024-09-16 22:08:00|
+----------+-------------------+
only showing top 5 rows
+--------------------+-----+
|      booking_status|count|
+--------------------+-----+
|     No Driver Found|10500|
|          Incomplete| 9000|
|           Completed|93000|
| Cancelled by Driver|27000|
|Cancelled by Cust...|10500|
+--------------------+-----+

+-------------+------------------+
|total_revenue|         avg_value|
+-------------+------------------+
|  5.1846183E7|508.29591176470586|
+-------------+------------------+



In [0]:
# Save cleaned data to Silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_rides")

In [0]:
# Verify the saved table
spark.read.table("silver_rides").show(5)

+----------+------------+-------------------+------------+------------+--------------------+----------------+--------+--------+---------------------+----------------------+-------------------+--------------------+----------------+-----------------+-------------+-------------+-------------+---------------+--------------+--------------------+-------------------+----+-----+-----------+----+------------+------------+
|      date|  booking_id|     booking_status| customer_id|vehicle_type|     pickup_location|   drop_location|avg_vtat|avg_ctat|cancelled_by_customer|customer_cancel_reason|cancelled_by_driver|driver_cancel_reason|incomplete_rides|incomplete_reason|booking_value|ride_distance|driver_rating|customer_rating|payment_method|      ingestion_time|   booking_datetime|year|month|day_of_week|hour|is_completed|is_cancelled|
+----------+------------+-------------------+------------+------------+--------------------+----------------+--------+--------+---------------------+-----------------